In [2]:
import requests as req
import pandas as pd

In [3]:
#se configura header y pagina a scrappear#

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "x-nextjs-data": "1"
}

response = req.get("https://www.fotmob.com/api/data/leagueseasondeepstats?id=38&season=27185&type=players&stat=goal_assist", 
             headers=headers)

print(response.status_code)



200


In [4]:
#obtenemos los datos en formato json y los imprimimos para ver su estructura#

data = response.json()
print(data.keys())
print(type(data))

dict_keys(['statsData', 'seasons', 'statsList', 'leagueDetails', 'type', 'teamName', 'currentSeasonId', 'currentStatName', 'substatTitle', 'substatLocalizedTitleId'])
<class 'dict'>


In [5]:
print(type(data['statsData']))
print(data['statsData'][0])

<class 'list'>
{'id': 1053284, 'teamId': 1954, 'name': 'Dejan Zukic', 'position': 85, 'substatValue': {'value': 7.9, 'format': 'fraction', 'fractions': 1}, 'statValue': {'name': 'goal_assist', 'value': 10, 'format': 'number', 'fractions': 0}, 'rank': 1, 'type': 'players'}


In [6]:
print(type(data['statsList']))
# Si es una lista, mira el primer elemento:
print(data['statsList'][13]) 
# O si es un diccionario, mira sus llaves:
# print(data['statsList'].keys())

<class 'list'>
{'name': 'total_att_assist', 'localizedTitleId': 'chances_created', 'title': 'Chances created', 'category': 'Attacking', 'localizedCategoryId': 'attack'}


In [7]:
#sacamos los datos de id, nombre y valor de cada jugador y los guardamos en un dataframe#

asistencias = []

for jugador in (data['statsData']):
    id = jugador.get('id', {})
    nombre = jugador.get('name', {})
    valor = jugador.get('substatValue', {}).get('value')

    df_fila = {'id': id, 'nombre': nombre, 'valor': valor}

    asistencias.append(df_fila)

df_asistencias = pd.DataFrame(asistencias)
print(df_asistencias)

          id              nombre  valor
0    1053284         Dejan Zukic    7.9
1    1276078      Matthias Seidl    6.8
2    1459265       Frans Krätzig    4.6
3     638127  Johannes Eggestein    4.4
4     978170         Tomi Horvat    6.2
..       ...                 ...    ...
154  1692197        Daniel Nunoo    0.2
155  1783320     Habib Coulibaly    0.2
156  1536477    Mahamadou Diarra    0.1
157  1714633     Emmanuel Ojukwu    0.0
158  1338941        Patrik Mijic    0.0

[159 rows x 3 columns]


In [28]:
stat_name = ["goals", "goal_assist", "expected_goals", "expected_goalsontarget", "total_scoring_att", "expected_assists_per_90","won_contest"
             , "accurate_long_balls", "poss_won_att_3rd", "defensive_contributions", "total_tackle", "interception",  "effective_clearance"]
league_id = [38,122]
season_id = [38611, 38000]
slug = ["bundesliga-players", "1-liga-players"]
build_id = "ZL64iOcIu6K0j4uv0G7J1"

endpoint = "https://www.fotmob.com/api/data/leagueseasondeepstats"

#f"https://www.fotmob.com/api/data/{build_id}/es/leagues/{league_id}/stats/season/{season_id}/players/{stat_name}/{slug}.json"

query_params = {
    "id": league_id,
    "season": season_id,
    "type": "players",
    "stat": stat_name,
    "slug": slug,
    "lng": "es"
}

datos_jugadores = []

for i in stat_name:
    query_params["stat"] = i
    
    response = req.get(endpoint, headers=headers, params=query_params)
    data = response.json()


    for jugador in data['statsData']:
        id = jugador.get('id', {})
        nombre = jugador.get('name', {})
        valor = jugador.get('statValue', {}).get('value')

        df_fila = {'id': id, 'nombre': nombre, 'valor': valor, 'stat': i, "league": slug[0]}

        datos_jugadores.append(df_fila)

df_datos_jugadores = pd.DataFrame(datos_jugadores)

    


In [32]:
print(df_datos_jugadores[df_datos_jugadores['nombre'] == "Wiktor Nowak"])

Empty DataFrame
Columns: [id, nombre, valor, stat, league]
Index: []


In [27]:
df_datos_jugadores_pivoted = df_datos_jugadores.pivot(index=['id', 'nombre'], columns='stat', values='valor').reset_index()
df_datos_jugadores_pivoted.isnull().sum()

stat
id                           0
nombre                       0
accurate_long_balls         62
defensive_contributions     38
effective_clearance         46
expected_assists_per_90     38
expected_goals              26
expected_goalsontarget      87
goal_assist                162
goals                      153
interception                81
poss_won_att_3rd           111
total_scoring_att           64
total_tackle                62
won_contest                 88
dtype: int64